**Navigation** : [Version Python](Z3-13b-UnsatCores-MUS-Python.ipynb) | [Version C#](Z3-13b-UnsatCores-MUS-CSharp.ipynb) | ← [13 - UNSAT cores](Z3-13-UnsatCores-Python.ipynb) | [README Z3-Python](README.md) | [14 - Bit-vectors >>](Z3-14-BitVectors-Overflow-Python.ipynb)

# Z3-Rust-13b — UNSAT cores : le MUS (sous-ensemble irréductible)

**Objectifs** : extraire un core et vérifier l'irréductibilité d'un MUS.
**Prérequis** : contraintes Z3 et notebook 13. **Durée** : 40 minutes.

*Compagnon « preuve minimale » du module [Z3-Python-13 — UNSAT cores](Z3-13-UnsatCores-Python.ipynb).*

Le notebook [13](Z3-13-UnsatCores-Python.ipynb) a établi le **noyau d'insatisfiabilité** (core) : `assert_and_track` étiquette chaque contrainte, `get_unsat_core()` isole celles qui conflictent. Ce compagnon 13b pousse là où le 13 s'arrête : d'abord un système où le conflit est **invisible à l'inspection** (aucune paire ne se contredit, l'insatisfiabilité émerge d'une combinaison), ensuite la limite de `get_unsat_core()` — Z3 ne garantit **pas** la minimalité — et l'algorithme **deletion-based** qui calcule le **MUS** (Minimal Unsatisfiable Subset) : le sous-ensemble *irréductible*, dont retirer n'importe quelle contrainte rend le système satisfiable.

> **Écueil d'API, mesuré sur le crate 0.23.** Le paramètre `unsat_core` ne se pose pas par `solver.set(...)` comme en pyz3 : le crate n'expose pas cette méthode. Il passe par `z3::Params` (`set_bool("unsat_core", true)`) puis `Solver::set_params`. Second écueil, plus discret : le crate n'implémente `IntoAst` que pour les valeurs **possédées**. `Int::le` attend une `Int`, jamais une `&Int` — il faut cloner la variable (`s_start[i + 1].clone()`). Une formule n'est pas une expression : `Ast::eq` et `Ast::ne` construisent l'égalité et la différence SMT, là où `==` et `!=` comparent deux valeurs Rust sans rien envoyer au solveur.

In [2]:
// Imports : le crate z3 (solveur SMT) et son API de modélisation.
// Le crate est lié dynamiquement à la bibliothèque Z3 installée sur le poste.
:dep z3 = { version = "0.23.0" }

use z3::ast::{Ast, Bool, Int};
use z3::{Params, Solver, SatResult};

println!("Imports OK : z3");
println!("z3 (lib) version : {}", z3::full_version());

Imports OK : z3


z3 (lib) version : 5.3.3.0 430d19e697e2ec9bad79ff210d41b4a7a5fcfa3e 430d19e


## 1. Le vrai test : un core invisible a l'inspection

Les exemples du notebook [13 - UNSAT cores](Z3-13-UnsatCores-Python.ipynb) partagent une limite : dans chacun, le conflit est **visible a l'oeil nu** -- deux contraintes se contredisent textuellement (`h == 12` vs `h != 12`, `x == 1` vs `x == 2`). On n'a pas vraiment besoin de Z3 pour les isoler.

C'est la que `get_unsat_core()` devient precieux : sur un systeme ou **aucune paire** de contraintes ne se contredit, mais ou l'insatisfiabilite **emerge d'une combinaison**. Cas concret, un projet de planification : quatre taches $T_0, T_1, T_2, T_3$ s'enchainent (chacune doit finir avant que la suivante commence), chacune dure 2 unites, et tout doit etre termine avant la date 7. Aucune contrainte prise isolement n'est absurde -- et pourtant le systeme est insatisfiable. Lequel des 11 ingredients faut-il relacher ?

In [3]:
// Planification surcontrainte : 4 taches enchainees (T0->T1->T2->T3), duree 2 chacune,
// a finir avant la date 7. Aucune contrainte isolee n'est absurde -> conflit structurel.
use z3::ast::{Ast, Bool, Int};
use z3::{Params, Solver, SatResult};

let duree: i64 = 2;
let date_limite: i64 = 7;
let s_start: Vec<Int> = (0..4).map(|i| Int::new_const(format!("start_T{}", i))).collect();

let s5 = Solver::new();
let mut params = Params::new();
params.set_bool("unsat_core", true);
s5.set_params(&params);

let mut labels: Vec<Bool> = Vec::new();

// Chaque tache : demarre apres 0, et finit avant la date limite.
// L'ordre des assertions est celui du jumeau Python (release puis deadline,
// tache par tache) : le core est renvoye par Z3 dans l'ordre d'assertion,
// donc les deux carnets affichent les memes etiquettes dans le meme ordre.
for i in 0..4 {
    let b = Bool::new_const(format!("release_T{}", i));
    s5.assert_and_track(s_start[i].ge(Int::from_i64(0)), &b);
    labels.push(b);
    let d = Bool::new_const(format!("deadline_T{}", i));
    s5.assert_and_track(
        Int::add(&[s_start[i].clone(), Int::from_i64(duree)]).le(Int::from_i64(date_limite)),
        &d,
    );
    labels.push(d);
}
// Enchainement : Ti finit avant que Ti+1 commence
for i in 0..3 {
    let b = Bool::new_const(format!("precedence_T{}_vers_T{}", i, i + 1));
    s5.assert_and_track(
        Int::add(&[s_start[i].clone(), Int::from_i64(duree)]).le(s_start[i + 1].clone()),
        &b,
    );
    labels.push(b);
}

let verdict = match s5.check() {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("Check : {}", verdict);

// Le core : les etiquettes des contraintes en conflit, dans l'ordre d'assertion.
let core = s5.get_unsat_core();
let noms_core: Vec<String> = core.iter().map(|c| format!("{}", c)).collect();
println!("Core Z3 = {{ {} }}", noms_core.join("  "));
println!("Taille du core : {} contraintes sur {}", noms_core.len(), labels.len());

// Les innocentes : tout ce qui n'est pas dans le core.
let noms: Vec<String> = labels.iter().map(|b| format!("{}", b)).collect();
let innocentes: Vec<String> = noms
    .iter()
    .filter(|n| !noms_core.contains(*n))
    .map(|n| n.to_string())
    .collect();
println!(
    "Contraintes innocentes ecartees ({}) : {}",
    innocentes.len(),
    innocentes.join(", ")
);

Check : unsat


Core Z3 = { release_T0  deadline_T3  precedence_T0_vers_T1  precedence_T1_vers_T2  precedence_T2_vers_T3 }


Taille du core : 5 contraintes sur 11


Contraintes innocentes ecartees (6) : deadline_T0, release_T1, deadline_T1, release_T2, deadline_T2, release_T3


### Lecture : le core isole la chaine, pas une paire

Le core observé contient **5 contraintes sur 11** : le `release_T0` (point de depart), les **trois precedences** qui enchainent $T_0 \to T_1 \to T_2 \to T_3$, et le `deadline_T3` (echeance finale). Les **6 autres** -- les bornes et echeances intermediaires `deadline_T0`, `release_T1`/`T2`/`T3`, `deadline_T1`/`T2` -- sont ecartees : la chaine les implique deja, elles ne sont pas necessaires au conflit.

Pourquoi ce core est **invisible a l'inspection** : aucune des 5 contraintes coupables ne contredit une autre textuellement. Il faut **tracer la chaine** -- `start_T0 >= 0`, puis `+2` a chaque precedence, donc `start_T1 >= 2`, `start_T2 >= 4`, `start_T3 >= 6`, puis `+2` de duree = `8 > 7` (la deadline) -- pour voir le conflit. Et ce core de 5 est le **SEUL possible** : retirer `release_T0` et la chaine flotte librement, retirer une precedence et la chaine se coupe, retirer `deadline_T3` et chaque echeance restante est tenable -- le core rendu n'est pas un choix parmi d'autres, c'est l'unique sous-ensemble irreductible, et les 6 contraintes ecartees le sont a bon droit.

C'est precisement le travail fastidieux qu'evite `get_unsat_core()` : parmi 11 contraintes, il isole chirurgicalement les 5 qui forment la preuve d'insatisfiabilite.

> Note de portage : en Python, la comparaison `s_start[i] >= 0` s'ecrit telle quelle. En Rust, elle devient `s_start[i].ge(Int::from_i64(0))` -- `ge` construit une **formule** SMT, la comparaison `>=` de Rust comparerait deux valeurs et ne produirait aucune formule. C'est la meme distinction que pour `Ast::eq` / `Ast::ne`.

### Exercice 1 — Irréductibilité membre par membre
Dans les huit contraintes suivantes, choisissez les étiquettes d'un MUS. Le garde teste UNSAT et chaque retrait ; une réponse vide ou redondante est refusée.

**Étapes** : former un sous-ensemble candidat, tester son insatisfiabilité, puis retirer chaque membre à tour de rôle pour vérifier son caractère essentiel.

In [4]:
// Exercice 1 : trouver un MUS parmi les huit contraintes.
// TODO etudiant :
// Etape 1 : w = Int::new_const("w_ex1") ; huit contraintes etiquetees :
//           nonneg (w >= 0), lower (w >= 4), upper (w + 4 <= 6), equal (w == 5),
//           bound (w <= 100), not7 (w != 7), floor (w >= -10), ceiling (w <= 50)
// Etape 2 : former un sous-ensemble candidat et tester son insatisfiabilite
// Etape 3 : retirer chaque membre a tour de role et verifier que le reste est SAT
// Indice : `Ast::eq(&w, Int::from_i64(5))` pour l'egalite, `Ast::ne` pour la
//           difference, `Int::add(&[w.clone(), Int::from_i64(4)])` pour la somme
//           (le crate expose `add` comme fonction variadique, pas comme methode) ;
//           le core d'un sous-ensemble se lit avec get_unsat_core().
let reponse_ex1: Option<Vec<String>> = None;  // TODO etudiant : etiquettes d'un MUS
println!("Exercice 1 a completer : trouvez un MUS parmi les huit contraintes");
let _ = &reponse_ex1;

Exercice 1 a completer : trouvez un MUS parmi les huit contraintes


## 2. Au-dela du core : le MUS (Minimal Unsatisfiable Subset)

Jusqu'ici, on a fait confiance a `get_unsat_core()` pour renvoyer un sous-ensemble
**minimal** de contraintes. Mais Z3 ne le **garantit pas** : le core est
*un sous-ensemble responsable du conflit*, pas necessairement le plus petit.
Sur l'arithmetique lineaire simple, Z3 minimise souvent bien ; sur des modeles
plus riches (graphe de precedences, contraintes transitives), il peut laisser
des contraintes **redondantes** dans le core.

Le **MUS** (Minimal Unsatisfiable Subset) est le sous-ensemble *irreductible* :
retirer n'importe laquelle de ses contraintes rend le systeme **satisfiable**.
L'algorithme **deletion-based** est simple et garantit cette irreductibilite :
on retire les contraintes une a une, et si le systeme reste UNSAT sans l'une
d'elles, c'est qu'elle etait superflue -- on la supprime definitivement.

Le **core** repond : « ces contraintes suffisent a etablir UNSAT ». Le **MUS** ajoute : « aucune de ces contraintes ne peut etre retiree sans retrouver SAT ». Cette garantie est une minimalite par inclusion, pas une recherche du cardinal minimum.

In [5]:
// Le MUS : l'algorithme deletion-based.
// On retire les contraintes une a une ; si le systeme reste UNSAT sans l'une d'elles,
// elle etait superflue -- on la supprime definitivement. Le resultat est irreductible
// par construction : retirer n'importe quel membre le rend satisfiable.
use z3::ast::{Ast, Bool, Int};
use z3::{Params, Solver, SatResult};

// L'affichage d'une liste d'etiquettes, dans le format du jumeau Python.
fn liste(items: &[String]) -> String {
    let interne: Vec<String> = items.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", interne.join(", "))
}

fn compute_mus(constraints: &[(Bool, String)]) -> Vec<(Bool, String)> {
    let mut mus: Vec<(Bool, String)> = constraints.to_vec();
    let mut i: usize = 0;
    while i < mus.len() {
        // Le candidat : le systeme SANS la contrainte i.
        let candidat: Vec<(Bool, String)> = mus
            .iter()
            .enumerate()
            .filter(|(j, _)| *j != i)
            .map(|(_, c)| (*c).clone())
            .collect();
        let mut s = Solver::new();
        let mut params = Params::new();
        params.set_bool("unsat_core", true);
        s.set_params(&params);
        for (formule, etiquette) in &candidat {
            let b = Bool::new_const(etiquette.clone());
            s.assert_and_track(formule.clone(), &b);
        }
        match s.check() {
            // Le systeme reste UNSAT -> la contrainte i etait superflue : on la supprime.
            SatResult::Unsat => mus = candidat,
            // Le systeme devient SAT -> la contrainte i est essentielle : on la garde.
            SatResult::Sat => i += 1,
            // Z3 repond `unknown` quand la theorie n'est pas decidee. Le jumeau
            // Python leve alors une erreur ; ici on ne peut pas faire echouer le
            // carnet (C.1), donc la contrainte est traitee comme essentielle --
            // elle est conservee, et le curseur avance. Sur l'arithmetique
            // entiere lineaire de ce carnet, `unknown` n'arrive jamais.
            SatResult::Unknown => {
                println!("ATTENTION : Z3 a renvoye unknown sur le candidat sans la contrainte {}", i);
                i += 1;
            }
        }
    }
    mus
}

// Ordonnancement de taches : 5 taches t0..t4 avec precedences (durees) + deadline.
// Les aretes transitives (ex. t0 -> t3) rendent certaines contraintes redondantes.
let t: Vec<Int> = (0..5).map(|k| Int::new_const(format!("t{}", k))).collect();
let precedences: Vec<(Bool, String)> = vec![
    (t[1].ge(Int::add(&[t[0].clone(), Int::from_i64(5)])), "e01".to_string()),   // t0 -> t1 (duree 5)
    (t[2].ge(Int::add(&[t[1].clone(), Int::from_i64(3)])), "e12".to_string()),   // t1 -> t2 (duree 3)
    (t[3].ge(Int::add(&[t[2].clone(), Int::from_i64(4)])), "e23".to_string()),   // t2 -> t3 (duree 4)
    (t[3].ge(Int::add(&[t[0].clone(), Int::from_i64(12)])), "e03".to_string()),  // t0 -> t3 transitif (5+3+4 = 12)
    (t[4].ge(Int::add(&[t[3].clone(), Int::from_i64(2)])), "e34".to_string()),   // t3 -> t4 (duree 2)
    (t[4].le(Int::add(&[t[0].clone(), Int::from_i64(10)])), "dead".to_string()), // deadline : chemin total = 14 > 10 -> UNSAT
];

// Core brut de Z3
let mut s_core = Solver::new();
let mut params = Params::new();
params.set_bool("unsat_core", true);
s_core.set_params(&params);
for (formule, etiquette) in &precedences {
    let b = Bool::new_const(etiquette.clone());
    s_core.assert_and_track(formule.clone(), &b);
}
let verdict = match s_core.check() {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("Ordonnancement : {}", verdict);

let core_z3: Vec<String> = s_core.get_unsat_core().iter().map(|c| format!("{}", c)).collect();
let mut core_triee = core_z3.clone();
core_triee.sort();
println!("Core Z3 (brut) : {} - taille {}", liste(&core_triee), core_triee.len());

// MUS irreductible
let mus = compute_mus(&precedences);
let mut mus_labels: Vec<String> = mus.iter().map(|(_, l)| l.clone()).collect();
mus_labels.sort();
println!("MUS (irreductible) : {} - taille {}", liste(&mus_labels), mus_labels.len());

let mut absentes: Vec<String> = core_z3
    .iter()
    .filter(|l| !mus_labels.contains(*l))
    .cloned()
    .collect();
absentes.sort();
println!("Etiquettes du core absentes du MUS : {}", liste(&absentes));
println!("-> Deux refutations peuvent etre irreductibles et de tailles differentes.");

Ordonnancement : unsat


Core Z3 (brut) : ['dead', 'e01', 'e12', 'e23', 'e34'] - taille 5


MUS (irreductible) : ['dead', 'e03', 'e34'] - taille 3


Etiquettes du core absentes du MUS : ['e01', 'e12', 'e23']


-> Deux refutations peuvent etre irreductibles et de tailles differentes.


**Lecture — deux preuves distinctes.** Le core brut peut suivre le chemin détaillé `e01`, `e12`, `e23`, `e34`, `dead`, alors que la suppression sur la liste entière trouve `e03`, `e34`, `dead`. Les trois premières arêtes impliquent ensemble `e03`, mais la réciproque est fausse : `e03` seule ne fixe pas les étapes intermédiaires.

Un core de cinq contraintes peut lui aussi être irréductible : le retrait de n'importe laquelle de ses contraintes le rend satisfiable. Le MUS de trois contraintes utilise une arête absente de ce core ; sa taille inférieure ne démontre donc pas que le core initial est réductible. Un MUS est minimal **par inclusion**, pas nécessairement de cardinal minimum. Il peut en exister plusieurs. Pour réparer la spécification entière, il faut vérifier à nouveau toutes les contraintes restantes : un autre conflit peut subsister.

> Note de portage : l'algorithme deletion-based est **le même** des deux côtés -- un curseur sur la liste, un test UNSAT du système amputé d'une contrainte. Seule l'API change : là où pyz3 écrit `solver.set("unsat_core", True)`, le crate passe par `Params` ; là où pyz3 garde les etiquettes dans une liste, le crate construit une etiquette `Bool::new_const(nom)` pour chaque assertion.

### Exercice 2 — Arête transitive contre chemin détaillé
Dans le modèle de trois tâches, choisissez un MUS. Comparez-le au chemin détaillé ; le core brut dépend du solveur et de l'ordre.

**Étapes** : identifier l'implication du chemin vers l'arête transitive, proposer les étiquettes, puis vérifier UNSAT et la satisfiabilité après chaque retrait.

In [6]:
// Exercice 2 : un MUS dans le modele de trois taches.
// TODO etudiant :
// Etape 1 : u0, u1, u2 = Int::new_const("u{i}_ex2") ; quatre contraintes etiquetees :
//           step01 (u1 >= u0 + 2), step12 (u2 >= u1 + 2), direct (u2 >= u0 + 4),
//           limit (u2 <= u0 + 3)
// Etape 2 : identifier l'implication du chemin vers l'arete transitive
// Etape 3 : verifier UNSAT sur le sous-ensemble, puis SAT apres chaque retrait
let reponse_ex2: Option<Vec<String>> = None;  // TODO etudiant : etiquettes d'un MUS
println!("Exercice 2 a completer : comparez le MUS au chemin detaille");
let _ = &reponse_ex2;

Exercice 2 a completer : comparez le MUS au chemin detaille


### Exercice 3 — Diagnostic et réparation
Retirez une étiquette du système de la section 2 puis vérifiez **toutes** les contraintes restantes. Retirer un membre d'un MUS ne suffit pas si un autre conflit subsiste.

**Étapes** : choisir une étiquette à retirer, reconstruire le système entier restant et contrôler SAT ; si le verdict reste UNSAT, expliquer quel conflit subsiste.

In [7]:
// Exercice 3 : reparer le systeme de la section 2 par le retrait d UNE etiquette.
// TODO etudiant :
// Etape 1 : choisir l'etiquette a retirer parmi les six du modele d'ordonnancement
// Etape 2 : reconstruire le systeme entier SANS elle (n'oublie pas les 5 autres)
// Etape 3 : verifier le verdict -- SAT confirme la reparation, UNSAT signale qu'un
//           autre conflit subsiste : il faut alors chercher un DEUXIEME retrait
let reponse_ex3: Option<String> = None;  // TODO etudiant : etiquette dont le retrait repare tout
println!("Exercice 3 a completer : retirez une etiquette et re-verifiez le systeme entier");
let _ = &reponse_ex3;

Exercice 3 a completer : retirez une etiquette et re-verifiez le systeme entier


## 3. Ce que ce notebook a demontre

Trois lecons au-dela du core brut :
1. **Là où ça compte** - sur un système où aucun conflit n'est visible à l'œil (section 1), le core isole la combinaison coupable parmi de nombreuses contraintes innocentes -- c'est là que le diagnostic devient indispensable, pas décoratif.

2. **Irréductibilité** - le MUS est le sous-ensemble dont **chaque** membre est essentiel : le retirer rend le système satisfiable. L'algorithme deletion-based le garantit constructivement, là où le core brut de Z3 peut contenir des contraintes redondantes.

3. **Deux preuves du même UNSAT** - le core brut et le MUS sont deux refutations différentes de la même insatisfiabilité ; le MUS trouvé ici est plus court (3 contraintes contre 5 dans le core observé), sans invalider l'irréductibilité éventuelle de ce dernier. Comparer les deux, c'est mesurer à quel point un diagnostic peut être resserré.

## 4. Exercices
Les exercices répartis dans le carnet portent sur l'irréductibilité, les chemins transitifs et la réparation du système entier.

## Conclusion
Un core suffit à prouver l'insatisfiabilité. Un MUS est irréductible par inclusion ; il peut en exister plusieurs, de tailles différentes. Une réparation proposée doit être vérifiée sur le système entier.

`get_unsat_core()` diagnostique **un conflit**. Énumérer **tous les MUS** est une tâche distincte, potentiellement plus coûteuse : un diagnostic isolé ne décrit pas tous les conflits possibles.

## À retenir
| Objet | Garantie |
|---|---|
| Core | Sous-ensemble insatisfiable |
| MUS | Sous-ensemble insatisfiable irréductible |

## Vérifiez votre compréhension
Pourquoi un MUS n'est-il pas forcément de cardinal minimum ?
<details><summary>Réponse</summary>Un autre MUS incomparable peut comporter moins de contraintes.</details>

## Pour aller plus loin
Poursuivez avec [Z3-14 - Bit-vectors](Z3-14-BitVectors-Overflow-Python.ipynb).